# PropertyScan — Real MediaFire Dataset Test (Colab T4)

**Full written guide:** `docs/COLAB_REAL_TEST_GUIDE.md`

| | |
|---|---|
| Dataset | MediaFire `archive.zip` (Kaggle 3D scan data) |
| Code | https://github.com/moizogg/3d-ai |
| **Working dir** | **`/content/3d-ai`** (repo root — NOT `revamped_code/`) |
| Progress | `[PROGRESS]` lines every **10 seconds** |

**Runtime → GPU (T4)** before you start.

## A1 — GPU check

In [ ]:
!nvidia-smi
import torch
print('CUDA', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

## A2 — Clone GitHub repo

Engine is at the **repo root**: `/content/3d-ai` should contain `propertyscan/`, `configs/`, `pyproject.toml`.

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/moizogg/3d-ai.git || true
%cd /content/3d-ai
!pwd
!ls -la
!ls propertyscan configs pyproject.toml

## A3 — Install package (must be in `/content/3d-ai`)

In [ ]:
%cd /content/3d-ai
!pip -q install -e ".[dev]"
!propertyscan version
!propertyscan doctor --profile colab_t4

## B1 — Download dataset (MediaFire)

If wget fails (403), **upload `archive.zip` manually** to `/content/data/`.

In [ ]:
from pathlib import Path
DATA = Path('/content/data')
DATA.mkdir(parents=True, exist_ok=True)
ZIP = DATA / 'archive.zip'
URL = 'https://download853.mediafire.com/rtmlk9ypemegyIXh1yTx9nFxnjYYGn5l5Y7jgmzy4HCb7cekTEY__64r_wSqKfc3Cns4Iohj56JzG_hKv6SpxVLLkM8y_ucaQCHgrNLuS14oFWQmjlp4IuwbXMKOG3aP7MOpotpk3DGZor2n-Kk5rH-m3LedwvDJXId-PhLfrTqM_Q/zknvd5nfvsygnsr/archive.zip'
!wget -c -O '{ZIP}' '{URL}' || true
print('exists', ZIP.exists(), 'MB', ZIP.stat().st_size/1e6 if ZIP.exists() else 0)

## B2 — Unzip + find INPUT

**Top-level** = only the first folder level (not all 400 images). Nested images still count in **Total images in zip tree**. If scattered, this cell flattens them.

In [ ]:
from pathlib import Path
import zipfile
import shutil

DATA = Path('/content/data')
ZIP = DATA / 'archive.zip'
EXTRACT = DATA / 'kaggle_scene'
EXTRACT.mkdir(parents=True, exist_ok=True)
assert ZIP.exists() and ZIP.stat().st_size > 1_000_000, 'Put archive.zip in /content/data first'

with zipfile.ZipFile(ZIP, 'r') as zf:
    zf.extractall(EXTRACT)

print('=== Top-level only (NOT all files in zip) ===')
for p in sorted(EXTRACT.iterdir())[:40]:
    print(' ', p.name, 'DIR' if p.is_dir() else f'file {p.stat().st_size}')

VIDEO_EXT = {'.mp4', '.mov', '.avi', '.mkv', '.webm'}
IMG_EXT = {'.jpg', '.jpeg', '.png', '.webp'}

all_images = [p for p in EXTRACT.rglob('*') if p.is_file() and p.suffix.lower() in IMG_EXT]
all_videos = [p for p in EXTRACT.rglob('*') if p.is_file() and p.suffix.lower() in VIDEO_EXT]
print('=== Full recursive counts ===')
print('Total images in zip tree:', len(all_images))
print('Total videos in zip tree:', len(all_videos))

def count_images_in_dir(d: Path) -> int:
    try:
        return sum(1 for f in d.iterdir() if f.is_file() and f.suffix.lower() in IMG_EXT)
    except Exception:
        return 0

best, best_n = None, 0
for d in [EXTRACT] + [p for p in EXTRACT.rglob('*') if p.is_dir()]:
    n = count_images_in_dir(d)
    if n > best_n:
        best, best_n = d, n

if all_videos:
    INPUT = sorted(all_videos, key=lambda p: -p.stat().st_size)[0]
    print('Using VIDEO:', INPUT)
elif best is not None and best_n >= max(8, int(0.8 * len(all_images)) if all_images else 8):
    INPUT = best
    print(f'Using image folder with {best_n} images:', INPUT)
elif len(all_images) >= 8:
    INPUT = DATA / 'all_frames_flat'
    if INPUT.exists():
        shutil.rmtree(INPUT)
    INPUT.mkdir(parents=True)
    for i, src in enumerate(sorted(all_images)):
        dest = INPUT / f'frame_{i:05d}{src.suffix.lower()}'
        shutil.copy2(src, dest)
    print(f'Images were nested/scattered. Flattened {len(all_images)} →', INPUT)
else:
    INPUT = None

print('INPUT =>', INPUT)
assert INPUT is not None

## C1 — SMOKE full pipeline (mock)

Always `%cd /content/3d-ai` first. Watch for `[PROGRESS] … every 10s`.

In [ ]:
%cd /content/3d-ai
from pathlib import Path
OUT = Path('/content/out_mock')
OUT.mkdir(exist_ok=True)
!propertyscan export --input "{INPUT}" --out "{OUT}" --profile colab_t4 --engine mock --train-backend mock
!ls -la "{OUT}"
print('scene.ply', (OUT/'scene.ply').exists())

## C2 — Install MASt3R / DUSt3R / Depth (real)

In [ ]:
!pip -q install transformers einops opencv-python-headless
!git clone --recursive https://github.com/naver/dust3r /content/dust3r
!pip -q install -e /content/dust3r
!git clone --recursive https://github.com/naver/mast3r /content/mast3r
!pip -q install -e /content/mast3r
%cd /content/3d-ai
!propertyscan doctor --profile colab_t4

## C3 — REAL MASt3R geometry

In [ ]:
%cd /content/3d-ai
from pathlib import Path
OUT_G = Path('/content/out_mast3r_geo')
OUT_G.mkdir(exist_ok=True)
!propertyscan geometry --input "{INPUT}" --out "{OUT_G}" --profile colab_t4 --engine mast3r
!ls -la "{OUT_G}"

## C4 — REAL export (mast3r + mock train)

In [ ]:
%cd /content/3d-ai
from pathlib import Path
OUT_R = Path('/content/out_real_export')
OUT_R.mkdir(exist_ok=True)
!propertyscan export --input "{INPUT}" --out "{OUT_R}" --profile colab_t4 --engine mast3r --train-backend mock
!ls -la "{OUT_R}"
import json
print(json.dumps(json.loads((OUT_R/'final_report.json').read_text()), indent=2)[:2000])

## C6 — Download results

In [ ]:
from google.colab import files
from pathlib import Path
import shutil
OUT = Path('/content/out_real_export')
bundle = Path('/content/propertyscan_results')
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir()
for name in ['scene.ply', 'property_scene.json', 'final_report.json', 'provenance.json']:
    src = OUT / name
    if src.exists():
        shutil.copy2(src, bundle / name)
!zip -r /content/propertyscan_results.zip /content/propertyscan_results
files.download('/content/propertyscan_results.zip')